# Tutorial 02 — The causal Transformer

[Lecture notes](../lectures/02_transformers.md) · [Slides](../slides/02_transformers.pptx) · [Exercise sheet](../exercises/02_transformers.md)

**40 minutes, CPU, NumPy only.** You will compute attention for two tokens by hand, run one decoder block with every shape visible, test that no position can see the future, and rotate queries and keys with RoPE. The weights are random numbers drawn with seed 42. Nothing here is a trained model.

| Minutes | Step |
|---|---|
| 0–8 | Attention for two tokens, checked against the board |
| 8–18 | One decoder block with explicit shapes |
| 18–25 | The causal-invariance test, and the shifted loss |
| 25–30 | RoPE: the score depends on the distance |
| 30–40 | Your turn: two functions with tests |

![Causal-mask matrix: each row can see its prefix only.](../data/causal-mask.svg)

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. Attention for two tokens

Take two tokens with Q = K = [[2, 0], [1, 1]] and V = [[1, 0], [0, 2]], so d_k = 2. Work out the attention weights and the output on paper, with and without the causal mask. Then run the cell.

In [2]:
def softmax(z):
    z=z-z.max(axis=-1,keepdims=True)
    e=np.exp(z)
    return e/e.sum(axis=-1,keepdims=True)

Q2 = K2 = np.array([[2., 0.], [1., 1.]]); V2 = np.array([[1., 0.], [0., 2.]])
scores2 = Q2 @ K2.T / np.sqrt(2)
future = np.triu(np.ones((2, 2), dtype=bool), 1)                 # True above the diagonal
A_open, A_causal = softmax(scores2), softmax(np.where(future, -np.inf, scores2))
print("scores\n", np.round(scores2, 3))
print("weights, no mask\n", np.round(A_open, 3), "\noutput\n", np.round(A_open @ V2, 3))
print("weights, causal mask\n", np.round(A_causal, 3), "\noutput\n", np.round(A_causal @ V2, 3))
assert np.allclose(A_open[0], [np.exp(np.sqrt(2)) / (1 + np.exp(np.sqrt(2))), 1 / (1 + np.exp(np.sqrt(2)))])
assert np.allclose(A_causal, [[1, 0], [0.5, 0.5]])

scores
 [[2.828 1.414]
 [1.414 1.414]]
weights, no mask
 [[0.804 0.196]
 [0.5   0.5  ]] 
output
 [[0.804 0.391]
 [0.5   1.   ]]
weights, causal mask
 [[1.  0. ]
 [0.5 0.5]] 
output
 [[1.  0. ]
 [0.5 1. ]]


The mask changes row 0 only. Without it, token 0 puts 0.196 of its weight on token 1, which comes later. Row 1 is the last position and may see everything, so the mask leaves it alone. Why does row 1 split its weight evenly?

## 2. One decoder block with explicit shapes

This is one pre-norm block with one head: 6 tokens, width d = 8, vocabulary V = 12. The function returns the logits and the attention weights. The upper triangle of the weights must be zero.

Before running: what are the shapes of `logits` and `a`?

In [3]:
T,d,V = 6,8,12
emb=rng.normal(0,.2,(V,d)); pos=rng.normal(0,.2,(T,d))
Wq,Wk,Wv,Wo=[rng.normal(0,.2,(d,d)) for _ in range(4)]
W1=rng.normal(0,.2,(d,2*d)); W2=rng.normal(0,.2,(2*d,d))
head=rng.normal(0,.2,(d,V))
def norm(x):
    return (x-x.mean(-1,keepdims=True))/np.sqrt(x.var(-1,keepdims=True)+1e-5)
def decoder(tokens,causal=True):
    x=emb[tokens]+pos[:len(tokens)]
    z=norm(x); q,k,v=z@Wq,z@Wk,z@Wv
    scores=q@k.T/np.sqrt(d)
    if causal: scores=np.where(np.triu(np.ones(scores.shape,dtype=bool),1),-np.inf,scores)
    a=softmax(scores)
    h=x+(a@v)@Wo
    h=h+np.maximum(norm(h)@W1,0)@W2
    return norm(h)@head,a
seq=np.array([1,3,5,7,2,4])
logits,a=decoder(seq)
print("logits shape",logits.shape,"attention\n",np.round(a,3))
assert np.allclose(a.sum(-1),1)
assert np.allclose(a[np.triu_indices(T,1)],0)

logits shape (6, 12) attention
 [[1.    0.    0.    0.    0.    0.   ]
 [0.53  0.47  0.    0.    0.    0.   ]
 [0.208 0.419 0.373 0.    0.    0.   ]
 [0.226 0.288 0.222 0.264 0.    0.   ]
 [0.213 0.169 0.283 0.085 0.251 0.   ]
 [0.257 0.109 0.134 0.075 0.275 0.151]]


`logits` is (6, 12): one row of vocabulary scores per position. `a` is (6, 6). Row 0 is [1, 0, 0, 0, 0, 0] whatever the weights are, because the first token can attend only to itself. Find the two lines in `decoder` that add to the residual stream `x` and `h`. Day 11 reads and edits that stream between layers.

## 3. Test causality

Plausible output proves nothing about the mask. Change tokens 3, 4 and 5 and compare the logits at positions 0, 1 and 2. Then switch the mask off and repeat.

Predict both differences before running.

In [4]:
changed=seq.copy(); changed[3:]=[9,8,10]
logits_changed,_=decoder(changed)
assert np.allclose(logits[:3],logits_changed[:3])
unmasked,_=decoder(seq,False); unmasked_changed,_=decoder(changed,False)
print("causal prefix difference",np.max(np.abs(logits[:3]-logits_changed[:3])))
print("unmasked prefix difference",np.max(np.abs(unmasked[:3]-unmasked_changed[:3])))
assert not np.allclose(unmasked[:3],unmasked_changed[:3])

causal prefix difference 0.0
unmasked prefix difference 0.18304620860607693


With the mask the difference is exactly 0. Without it the early logits move, so those positions were reading tokens they are meant to predict. This test needs no trained weights and no labels. Run it on every decoder you write or modify.

## 4. Shift the targets by one

Teacher forcing: position t predicts token t + 1. The last position has no target in this sequence. The loop repeats the invariance test at every cut point.

The weights are random. What loss do you expect?

In [5]:
pred=softmax(logits[:-1]); targets=seq[1:]
loss=-np.log(pred[np.arange(len(targets)),targets]).mean()
print("next-token loss",float(loss),"uniform reference",float(np.log(V)))
for cut in range(1,T):
    alt=seq.copy(); alt[cut:]=(alt[cut:]+1)%V
    assert np.allclose(decoder(seq)[0][:cut],decoder(alt)[0][:cut])

next-token loss 2.765809801061805 uniform reference 2.4849066497880004


The loss is 2.77, a little above ln 12 = 2.485, the loss of a uniform guess over 12 tokens. An untrained model should start close to ln V. A first-step loss far below ln V is a warning that the model sees its targets.

## 5. RoPE: the score depends on the distance

Rotary position embedding rotates each pair of coordinates of a query or key by an angle proportional to its position. The dot product of a rotated query and a rotated key then depends on how far apart they are.

Predict which of the four scores below are equal.

In [6]:
def rope(x, position, base=10000.0):
    """Rotate consecutive coordinate pairs of a vector by position * theta_i."""
    half = len(x) // 2
    theta = position * base ** (-np.arange(half) / half)
    x1, x2 = x[0::2], x[1::2]
    out = np.empty_like(x)
    out[0::2] = x1 * np.cos(theta) - x2 * np.sin(theta)
    out[1::2] = x1 * np.sin(theta) + x2 * np.cos(theta)
    return out

q_vec, k_vec = rng.normal(size=d), rng.normal(size=d)
for m, n in [(3, 1), (10, 8), (50, 48), (3, 2)]:
    print(f"query at {m:2d}, key at {n:2d}: score {rope(q_vec, m) @ rope(k_vec, n):+.4f}")
assert np.isclose(rope(q_vec, 3) @ rope(k_vec, 1), rope(q_vec, 50) @ rope(k_vec, 48))
assert np.isclose(np.linalg.norm(rope(q_vec, 7)), np.linalg.norm(q_vec))       # a rotation keeps the length

query at  3, key at  1: score -2.8705
query at 10, key at  8: score -2.8705
query at 50, key at 48: score -2.8705
query at  3, key at  2: score -2.6334


The first three scores are equal: each pair is 2 positions apart. The fourth pair is 1 apart and scores differently. RoPE adds no parameters and leaves vector lengths unchanged. It gives no promise about sequences longer than those seen in training.

## 6. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. Multi-head attention with shape checks.** `x` has shape (B, T, d). Project to queries, keys and values with `Wq`, `Wk`, `Wv`. Split each into `n_heads` heads of width d / n_heads, giving (B, H, T, d/H). Scale the scores by the square root of the head width. `mask` is boolean, broadcastable to (B, H, T, T), and True where attention is allowed. Merge the heads back to (B, T, d) and multiply by `Wo`. Return the output and the weights. Assert the shapes inside your function.

In [ ]:
# YOUR TURN
def multi_head_attention(x, Wq, Wk, Wv, Wo, n_heads, mask):
    """Return (output of shape (B, T, d), attention weights of shape (B, H, T, T))."""
    raise NotImplementedError("your turn")

In [8]:
causal = np.tril(np.ones((T, T), dtype=bool))
z = norm(emb[seq] + pos)
out1, a1 = multi_head_attention(z[None], Wq, Wk, Wv, Wo, 1, causal)
assert out1.shape == (1, T, d) and a1.shape == (1, 1, T, T)
assert np.allclose(a1[0, 0], a) and np.allclose(out1[0], (a @ (z @ Wv)) @ Wo)     # one head reproduces section 2

X = rng.normal(size=(2, T, d))                                 # a batch of two sequences
out, attn = multi_head_attention(X, Wq, Wk, Wv, Wo, 4, causal)
assert out.shape == (2, T, d) and attn.shape == (2, 4, T, T)
assert np.allclose(attn.sum(-1), 1) and np.allclose(attn[..., np.triu_indices(T, 1)[0], np.triu_indices(T, 1)[1]], 0)
X_alt = X.copy(); X_alt[:, 3:] += 1.0                          # change the future
assert np.allclose(out[:, :3], multi_head_attention(X_alt, Wq, Wk, Wv, Wo, 4, causal)[0][:, :3])

lengths = np.array([6, 4])                                     # the second sequence has two padding tokens
not_pad = np.arange(T)[None, :] < lengths[:, None]             # (B, T)
both = causal & not_pad[:, None, None, :]                      # (B, 1, T, T): causal and key-is-real
_, attn_pad = multi_head_attention(X, Wq, Wk, Wv, Wo, 2, both)
assert np.allclose(attn_pad[1, :, :, 4:], 0) and np.allclose(attn_pad.sum(-1), 1)
print("weight on padding keys:", float(attn_pad[1, :, :, 4:].max()), " shapes:", out.shape, attn.shape)
print("A passed")

weight on padding keys: 0.0  shapes: (2, 6, 8) (2, 4, 6, 6)
A passed


**B. Count the parameters of a block.** A pre-norm block has four d × d attention matrices, an MLP with matrices d × d_ff and d_ff × d, and two LayerNorms with a gain and a bias vector each. With `bias=True` every attention matrix and both MLP matrices also have a bias vector.

In [ ]:
# YOUR TURN
def block_params(d_model, d_ff, bias=True):
    """Number of trainable parameters in one pre-norm Transformer block with LayerNorm."""
    raise NotImplementedError("your turn")

In [10]:
assert block_params(8, 16, bias=False) == sum(w.size for w in [Wq, Wk, Wv, Wo, W1, W2]) + 4 * 8
assert block_params(768, 3072) == 7_087_872                      # one block of GPT-2 small
gpt2_small = 12 * block_params(768, 3072) + 50257 * 768 + 1024 * 768 + 2 * 768
print("GPT-2 small: 12 blocks + token and position embeddings + final norm =", f"{gpt2_small:,}")
assert gpt2_small == 124_439_808
dm, hd, kv, ff = 4096, 32, 8, 14336                              # Llama-3.1-8B: width, heads, KV heads, MLP width
llama_block = 2 * dm * dm + 2 * dm * (dm // hd * kv) + 3 * dm * ff + 2 * dm
llama = 32 * llama_block + 2 * 128256 * dm + dm
print("Llama-3.1-8B: one block", f"{llama_block:,}", " whole model", f"{llama:,}")
print("B passed")

GPT-2 small: 12 blocks + token and position embeddings + final norm = 124,439,808
Llama-3.1-8B: one block 218,112,000  whole model 8,030,261,248
B passed


The count for GPT-2 small matches its published 124 million. The Llama line uses grouped-query attention (key and value matrices of width 8 × 128 in place of 32 × 128), a three-matrix SwiGLU MLP, RMSNorm gains, and separate input and output embedding tables of 128,256 rows. It gives 8.03 billion. The MLP holds two thirds of a GPT-2 block and four fifths of a Llama block.

## Discussion

1. Someone masks by zeroing weights after the softmax and does not renormalise. What is wrong with the rows?
2. A decoder trains on all positions of a sequence in one pass. Why does that not leak the future?
3. An additive mask of zeros is applied by mistake. Training loss falls fast. Which test catches the bug?

## From the toy to your task

Open the configuration file of the model your team plans to use and find the width, the number of layers, the number of heads and the MLP width. The exercise sheet asks you to count its parameters before you download it.

Submit this notebook with both tests passing and a 150–250-word note: what the invariance test showed, one bug it would miss, and the parameter count of your team's model.